# OCT final patch — subtype matching + deterministic rerun + corrected Grad-CAM

This notebook is the **final minimal-compute patch** for the corrected OCT experiment.

It is designed to use the **output of `oct-retina-improved-final.ipynb` as a Kaggle input**, so the already-audited cohort and fixed fold/test/validation manifests are reused rather than rebuilt. The raw Kermany OCT dataset must also be attached because model training and Grad-CAM still require the image files.

## What is changed

1. **Pathology-subtype matching**
   - The rigorous train, validation and fixed test manifests from the previous corrected run are preserved.
   - The controlled-leak train is rebuilt by **Disease-matched swaps**:
     - `NORMAL ↔ NORMAL`
     - `CNV ↔ CNV`
     - `DME ↔ DME`
     - `DRUSEN ↔ DRUSEN`
   - Therefore training size, binary counts **and all four pathology-subtype counts** are identical between arms.

2. **Deterministic paired rerun**
   - Both rigorous and controlled-leak arms are rerun.
   - `cudnn.benchmark=False`, `cudnn.deterministic=True`, seeded DataLoader ordering and deterministic-algorithm settings are enabled.
   - The rigorous arm is intentionally **not** reused from the previous predictions, because doing so would not produce a fully deterministic final paired experiment.

3. **Corrected XAI**
   - Quantitative Grad-CAM is run only for **ResNet-50** to minimize compute.
   - The target is the **true binary class** for both paired models, keeping the attribution target identical.
   - Border Activation Ratio (outer 10% border) is calculated for **all primary OOF test images**.
   - The rigorous-vs-controlled-leak BR difference is summarized with patient-cluster bootstrap CIs.
   - A fixed, seeded, disease-stratified set of 12 cases is selected **before model results are inspected** for visual overlays.
   - No old contaminated checkpoint or old XAI output is reused.

## What is deliberately not changed

The previous corrected experiment's **cohort, outer fixed test assignments, rigorous train manifests and safe validation manifests are retained**. This minimizes methodological drift and avoids unnecessary preprocessing. The final patch changes only what is required to remove the remaining subtype confound and regenerate the paired models deterministically.

## Kaggle inputs

Attach:

1. **Output of the executed `oct-retina-improved-final.ipynb` notebook** as an input.
   - The notebook can consume either the exposed `oct_corrected_leakage_audit/` folder or `OCT_Corrected_Leakage_Audit_Results.zip`.
2. The same **Kermany OCT2017 / kermany2018** image dataset used previously.

Then enable a GPU and **Run All**.

The final archive is written to:

`/kaggle/working/OCT_FinalPatch_Results.zip`

In [1]:
# =========================
# 1. Configuration + deterministic environment
# =========================
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from pathlib import Path
import re, sys, json, math, time, random, hashlib, shutil, zipfile, platform, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, recall_score,
    brier_score_loss, log_loss
)

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0     # strongest cross-run DataLoader determinism
FOLDS = 5
EPOCHS = 12
PATIENCE = 3
LR = 1e-4
WEIGHT_DECAY = 1e-4
LEAK_FRACTION = 1.0
BOOTSTRAPS = 2000
PERMUTATIONS = 2000
RUN_ARCHITECTURES = ["ResNet50", "ConvNeXtTiny"]
PRETRAINED = True

# Corrected XAI: quantitative Grad-CAM only on ResNet-50.
XAI_ARCHITECTURE = "ResNet50"
XAI_BORDER_FRAC = 0.10
XAI_VISUAL_CASES_PER_DISEASE = 3  # 4 diseases x 3 = 12 fixed cases

# Normally leave blank: auto-discovers previous notebook output mounted under /kaggle/input.
PRIOR_OUTPUT_OVERRIDE = ""
DATA_ROOT_OVERRIDE = ""

BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUT = BASE / "oct_final_patch"
PRED_DIR = OUT / "predictions"
SPLIT_DIR = OUT / "splits"
XAI_DIR = OUT / "xai"
FIG_DIR = OUT / "figures"
for p in [OUT, PRED_DIR, SPLIT_DIR, XAI_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

# Deterministic algorithms: warn_only avoids a crash if a platform-specific operation has
# no deterministic implementation, while the warning is captured in the reproducibility record.
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except TypeError:
    torch.use_deterministic_algorithms(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU execution will be very slow; use a Kaggle/Colab GPU.")

versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
    "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
    "deterministic_algorithms": bool(torch.are_deterministic_algorithms_enabled()),
}
print(json.dumps(versions, indent=2))

Device: cuda
GPU: Tesla T4
{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "cuda": "12.8",
  "device": "Tesla T4",
  "cudnn_deterministic": true,
  "cudnn_benchmark": false,
  "deterministic_algorithms": true
}


In [2]:
# =========================
# 2. Locate previous corrected OCT output
# =========================
REQUIRED_PRIOR = [
    "cohort_manifest.csv",
    "splits/fold1_rig_train.csv",
    "splits/fold1_val.csv",
    "splits/fold1_test.csv",
]

def valid_prior_root(p):
    p = Path(p)
    return p.exists() and all((p / x).exists() for x in REQUIRED_PRIOR)

def discover_prior():
    if PRIOR_OUTPUT_OVERRIDE:
        p = Path(PRIOR_OUTPUT_OVERRIDE)
        if valid_prior_root(p):
            return p
        raise FileNotFoundError(f"PRIOR_OUTPUT_OVERRIDE is not a valid prior output root: {p}")

    bases = [Path("/kaggle/input"), Path("/mnt/data"), Path.cwd()]
    candidates = []

    for base in bases:
        if not base.exists():
            continue

        # Direct folder output.
        for p in base.rglob("cohort_manifest.csv"):
            root = p.parent
            if valid_prior_root(root):
                candidates.append(root)

        # Results ZIP output.
        for z in base.rglob("OCT_Corrected_Leakage_Audit_Results.zip"):
            extract_root = BASE / "_prior_oct_results"
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z, "r") as fh:
                fh.extractall(extract_root)
            for p in extract_root.rglob("cohort_manifest.csv"):
                root = p.parent
                if valid_prior_root(root):
                    candidates.append(root)

    # Deduplicate and prefer roots containing the exact expected folder name.
    uniq = []
    seen = set()
    for p in candidates:
        rp = str(p.resolve())
        if rp not in seen:
            uniq.append(p)
            seen.add(rp)

    if not uniq:
        raise FileNotFoundError(
            "Previous corrected OCT output was not found. On Kaggle add the OUTPUT of "
            "`oct-retina-improved-final.ipynb` as an input. It must expose either "
            "`oct_corrected_leakage_audit/cohort_manifest.csv` plus `splits/`, or "
            "`OCT_Corrected_Leakage_Audit_Results.zip`."
        )

    uniq.sort(key=lambda p: (0 if p.name == "oct_corrected_leakage_audit" else 1, len(str(p))))
    print("Prior-output candidates:")
    for p in uniq[:10]:
        print(" ", p)
    return uniq[0]

PRIOR = discover_prior()
print("Using prior corrected output:", PRIOR)

cohort = pd.read_csv(PRIOR / "cohort_manifest.csv")
required_cols = {"row_id", "Patient_ID", "Label", "Disease", "sample_key", "filepath"}
missing = required_cols - set(cohort.columns)
if missing:
    raise RuntimeError(f"Prior cohort manifest lacks required columns: {sorted(missing)}")

cohort["Patient_ID"] = cohort["Patient_ID"].astype(str)
cohort["Disease"] = cohort["Disease"].astype(str).str.upper()
cohort["row_id"] = cohort["row_id"].astype(int)
cohort["Label"] = cohort["Label"].astype(int)

print(f"Prior cohort: {len(cohort)} images | {cohort.Patient_ID.nunique()} patients")
print(cohort.Disease.value_counts().to_string())

Prior-output candidates:
  /kaggle/working/_prior_oct_results/oct_corrected_leakage_audit
  /kaggle/input/notebooks/sunzilkhandaker/oct-retina-improved-final/oct_corrected_leakage_audit
Using prior corrected output: /kaggle/working/_prior_oct_results/oct_corrected_leakage_audit
Prior cohort: 1200 images | 855 patients
Disease
NORMAL    600
CNV       406
DME       109
DRUSEN     85


In [3]:
# =========================
# 3. Resolve raw OCT images and remap stale prior paths if necessary
# =========================
CLASSES = {"CNV", "DME", "DRUSEN", "NORMAL"}
IMG_EXTS = {".jpeg", ".jpg", ".png"}
FNAME_RE = re.compile(r"^(CNV|DME|DRUSEN|NORMAL)-([^-]+)-(.+?)\.(jpeg|jpg|png)$", re.I)

def candidate_oct_roots(search_base):
    search_base = Path(search_base)
    out = []
    if not search_base.exists():
        return out
    for p in search_base.rglob("*"):
        if not p.is_dir():
            continue
        try:
            children = {x.name.strip().lower(): x for x in p.iterdir() if x.is_dir()}
        except Exception:
            continue
        if "train" not in children:
            continue
        tr_classes = {x.name.strip().upper() for x in children["train"].iterdir() if x.is_dir()}
        if CLASSES.issubset(tr_classes):
            n = sum(
                1 for x in p.rglob("*")
                if x.is_file() and x.suffix.lower() in IMG_EXTS and FNAME_RE.match(x.name)
            )
            out.append((n, p))
    return sorted(out, key=lambda z: (-z[0], len(str(z[1]))))

def resolve_oct_root():
    if DATA_ROOT_OVERRIDE:
        p = Path(DATA_ROOT_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(DATA_ROOT_OVERRIDE)
        return p

    allc = []
    for b in [Path("/kaggle/input"), Path("/mnt/data"), Path.cwd()]:
        allc.extend(candidate_oct_roots(b))
    if not allc:
        raise FileNotFoundError(
            "Kermany OCT image dataset not found. Attach the same kermany2018/OCT2017 dataset "
            "in addition to the previous notebook output."
        )

    # Avoid accidentally selecting the prior-results folder; it contains no full image hierarchy anyway.
    print("OCT dataset candidates:")
    for n, p in allc[:10]:
        print(f"  {n:>7,} images  {p}")
    return allc[0][1]

ROOT = resolve_oct_root()
print("Image root:", ROOT)

# Build sample_key -> filepath index once.
image_index = {}
for f in ROOT.rglob("*"):
    if not f.is_file() or f.suffix.lower() not in IMG_EXTS:
        continue
    m = FNAME_RE.match(f.name)
    if not m:
        continue
    key = f"{m.group(1).upper()}-{m.group(2)}-{m.group(3)}"
    image_index.setdefault(key, str(f))

missing_keys = [k for k in cohort.sample_key.astype(str) if k not in image_index]
if missing_keys:
    raise RuntimeError(
        f"{len(missing_keys)} prior cohort images could not be mapped into the attached raw dataset. "
        f"First examples: {missing_keys[:5]}"
    )

cohort["filepath"] = cohort.sample_key.astype(str).map(image_index)
assert cohort.filepath.notna().all()
print("All prior cohort rows remapped to current raw-image paths.")

OCT dataset candidates:
   84,484 images  /kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 
   84,484 images  /kaggle/input/datasets/paultimothymooney/kermany2018/oct2017/OCT2017 
        0 images  /kaggle/input/datasets/paultimothymooney/kermany2018/oct2017/__MACOSX/OCT2017 
Image root: /kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 
All prior cohort rows remapped to current raw-image paths.


In [4]:
# =========================
# 4. Add decoded-content fingerprint audit (low cost, no retraining)
# =========================
def decoded_fingerprint(path, side=128):
    with Image.open(path) as im:
        im = im.convert("L")
        original_size = im.size
        im = im.resize((side, side), Image.Resampling.BILINEAR)
        arr = np.asarray(im, dtype=np.uint8)
    h = hashlib.sha256()
    h.update(str(original_size).encode("utf-8"))
    h.update(arr.tobytes())
    return h.hexdigest()

fp_cache = OUT / "decoded_fingerprints.csv"
if fp_cache.exists():
    q = pd.read_csv(fp_cache)
    if set(q.sample_key.astype(str)) == set(cohort.sample_key.astype(str)):
        cohort = cohort.drop(columns=["decoded_fp"], errors="ignore").merge(
            q[["sample_key", "decoded_fp"]], on="sample_key", how="left"
        )

if "decoded_fp" not in cohort.columns or cohort.decoded_fp.isna().any():
    print("Computing decoded-content fingerprints for 1,200-image cohort...")
    cohort["decoded_fp"] = [decoded_fingerprint(p) for p in cohort.filepath]
    cohort[["sample_key", "decoded_fp"]].to_csv(fp_cache, index=False)

decoded_dups = int(cohort.duplicated("decoded_fp").sum())
print("Decoded-content duplicates inside prior cohort:", decoded_dups)
if decoded_dups:
    raise RuntimeError(
        "Decoded-content duplicates are present in the frozen prior cohort. "
        "Do not silently alter the cohort in this patch; inspect these rows before proceeding."
    )

Computing decoded-content fingerprints for 1,200-image cohort...
Decoded-content duplicates inside prior cohort: 0


In [5]:
# =========================
# 5. Reuse rigorous/validation/test manifests; rebuild ONLY controlled-leak manifests
#    with exact pathology-subtype matching
# =========================
DISEASE_ORDER = ["NORMAL", "CNV", "DME", "DRUSEN"]

def remap_manifest_paths(df):
    df = df.copy()
    df["Patient_ID"] = df["Patient_ID"].astype(str)
    df["Disease"] = df["Disease"].astype(str).str.upper()
    df["row_id"] = df["row_id"].astype(int)
    df["Label"] = df["Label"].astype(int)
    df["filepath"] = df.sample_key.astype(str).map(image_index)
    df["decoded_fp"] = df.sample_key.astype(str).map(
        cohort.set_index("sample_key")["decoded_fp"].to_dict()
    )
    if df.filepath.isna().any():
        raise RuntimeError("A prior split manifest contains images absent from the attached raw dataset.")
    return df

def sample_n(df, n, seed):
    if n <= 0:
        return df.iloc[0:0].copy()
    return df.sample(n=n, random_state=seed, replace=False)

FOLD_DATA = {}
audit_rows = []

for fold in range(1, FOLDS + 1):
    rig = remap_manifest_paths(pd.read_csv(PRIOR / "splits" / f"fold{fold}_rig_train.csv"))
    val = remap_manifest_paths(pd.read_csv(PRIOR / "splits" / f"fold{fold}_val.csv"))
    test = remap_manifest_paths(pd.read_csv(PRIOR / "splits" / f"fold{fold}_test.csv"))

    test_ids = set(test.row_id.astype(int))
    test_patients = set(test.Patient_ID.astype(str))
    test_decoded = set(test.decoded_fp.astype(str))

    # Candidate pool is reconstructed from the frozen cohort: non-test rows belonging
    # to test patients. Exact/decoded test content remains forbidden.
    cand = cohort[
        (~cohort.row_id.astype(int).isin(test_ids))
        & (cohort.Patient_ID.astype(str).isin(test_patients))
        & (~cohort.decoded_fp.astype(str).isin(test_decoded))
    ].copy().reset_index(drop=True)

    rng_seed = SEED + 10000 * fold
    remove_ids = []
    leak_parts = []

    for di, disease in enumerate(DISEASE_ORDER):
        dc = cand[cand.Disease == disease].copy()
        base = rig[rig.Disease == disease].copy()

        n = int(round(len(dc) * LEAK_FRACTION))
        n = min(n, len(dc), max(len(base) - 1, 0))
        chosen = sample_n(dc, n, rng_seed + 100 + di)
        removed = sample_n(base, n, rng_seed + 200 + di)

        leak_parts.append(chosen)
        remove_ids.extend(removed.row_id.astype(int).tolist())

    leak = rig[~rig.row_id.astype(int).isin(remove_ids)].copy()
    leak = pd.concat([leak] + leak_parts, ignore_index=True)
    leak = leak.sample(frac=1, random_state=rng_seed + 999).reset_index(drop=True)

    # Exposure variables.
    sib = leak.Patient_ID.astype(str).value_counts()
    same_dis = leak.groupby(["Patient_ID", "Disease"]).size()
    test["n_train_siblings"] = test.Patient_ID.astype(str).map(sib).fillna(0).astype(int)
    test["n_same_disease_siblings"] = [
        int(same_dis.get((str(p), str(d)), 0))
        for p, d in zip(test.Patient_ID, test.Disease)
    ]

    def ids(x): return set(x.row_id.astype(int))
    def pats(x): return set(x.Patient_ID.astype(str))
    def decs(x): return set(x.decoded_fp.astype(str))

    # Integrity assertions.
    assert ids(test).isdisjoint(ids(rig))
    assert ids(test).isdisjoint(ids(leak))
    assert decs(test).isdisjoint(decs(rig))
    assert decs(test).isdisjoint(decs(leak))
    assert pats(test).isdisjoint(pats(rig))
    assert pats(test).isdisjoint(pats(val))
    assert pats(val).isdisjoint(pats(rig))
    assert pats(val).isdisjoint(pats(leak))
    assert len(rig) == len(leak)

    rig_d = rig.Disease.value_counts().reindex(DISEASE_ORDER, fill_value=0).to_dict()
    leak_d = leak.Disease.value_counts().reindex(DISEASE_ORDER, fill_value=0).to_dict()
    assert rig_d == leak_d, f"Fold {fold}: pathology-subtype counts differ: {rig_d} vs {leak_d}"

    audit = {
        "fold": fold,
        "test_n": len(test),
        "test_patients": test.Patient_ID.nunique(),
        "rig_train_n": len(rig),
        "leak_train_n": len(leak),
        "val_n": len(val),
        "rig_train_test_patient_overlap": len(pats(rig) & pats(test)),
        "leak_train_test_patient_overlap": len(pats(leak) & pats(test)),
        "rig_train_test_decoded_overlap": len(decs(rig) & decs(test)),
        "leak_train_test_decoded_overlap": len(decs(leak) & decs(test)),
        "test_exposure_pct": float((test.n_train_siblings > 0).mean()),
        "same_disease_exposure_pct": float((test.n_same_disease_siblings > 0).mean()),
        "rig_NORMAL": rig_d["NORMAL"],
        "rig_CNV": rig_d["CNV"],
        "rig_DME": rig_d["DME"],
        "rig_DRUSEN": rig_d["DRUSEN"],
        "leak_NORMAL": leak_d["NORMAL"],
        "leak_CNV": leak_d["CNV"],
        "leak_DME": leak_d["DME"],
        "leak_DRUSEN": leak_d["DRUSEN"],
    }

    FOLD_DATA[fold] = {"rig": rig, "leak": leak, "val": val, "test": test}
    audit_rows.append(audit)

    for role, frame in [("rig_train", rig), ("leak_train", leak), ("val", val), ("test", test)]:
        x = frame.copy()
        x["role"] = role
        x["fold"] = fold
        x.to_csv(SPLIT_DIR / f"fold{fold}_{role}.csv", index=False)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT / "subtype_matched_leakage_audit_by_fold.csv", index=False)
print(audit_df.to_string(index=False))

assert (audit_df.rig_train_test_patient_overlap == 0).all()
assert (audit_df.rig_train_test_decoded_overlap == 0).all()
assert (audit_df.leak_train_test_decoded_overlap == 0).all()
for disease in DISEASE_ORDER:
    assert (audit_df[f"rig_{disease}"] == audit_df[f"leak_{disease}"]).all()

print("\nPATHOLOGY-SUBTYPE MATCHING AUDIT: PASSED")

 fold  test_n  test_patients  rig_train_n  leak_train_n  val_n  rig_train_test_patient_overlap  leak_train_test_patient_overlap  rig_train_test_decoded_overlap  leak_train_test_decoded_overlap  test_exposure_pct  same_disease_exposure_pct  rig_NORMAL  rig_CNV  rig_DME  rig_DRUSEN  leak_NORMAL  leak_CNV  leak_DME  leak_DRUSEN
    1     240            218          652           652    167                               0                               78                               0                                0           0.391667                   0.362500         367      185       50          50          367       185        50           50
    2     240            224          644           644    159                               0                               90                               0                                0           0.433333                   0.383333         362      179       64          39          362       179        64           39
    3     240      

In [6]:
# =========================
# 6. Fixed, prediction-independent XAI visualization case selection
# =========================
# Choose cases before any model output is available: 3 seeded images per Disease,
# drawn from the frozen cohort. Each image belongs to exactly one test fold.
vis_parts = []
for i, disease in enumerate(DISEASE_ORDER):
    q = cohort[cohort.Disease == disease].copy()
    n = min(XAI_VISUAL_CASES_PER_DISEASE, len(q))
    vis_parts.append(q.sample(n=n, random_state=SEED + 700 + i))

visual_cases = pd.concat(vis_parts, ignore_index=True)
visual_case_ids = set(visual_cases.row_id.astype(int))
visual_cases[["row_id", "sample_key", "Disease", "Patient_ID"]].to_csv(
    XAI_DIR / "preselected_visual_cases.csv", index=False
)

print("Preselected XAI visualization cases:")
print(visual_cases[["row_id", "sample_key", "Disease", "Patient_ID"]].to_string(index=False))

Preselected XAI visualization cases:
 row_id        sample_key Disease Patient_ID
    318 NORMAL-7984958-12  NORMAL    7984958
    723  NORMAL-1122061-1  NORMAL    1122061
    234  NORMAL-5434564-7  NORMAL    5434564
     52    CNV-9811594-41     CNV    9811594
   1123   CNV-8728968-133     CNV    8728968
    477   CNV-9439639-109     CNV    9439639
    640     DME-7939555-5     DME    7939555
    805    DME-7846574-37     DME    7846574
    554     DME-8200056-5     DME    8200056
    708 DRUSEN-1146923-29  DRUSEN    1146923
    127  DRUSEN-364469-45  DRUSEN     364469
    734 DRUSEN-1786810-23  DRUSEN    1786810


In [7]:
# =========================
# 7. OCT data pipeline
# =========================
IM_MEAN = (0.485, 0.456, 0.406)
IM_STD = (0.229, 0.224, 0.225)

TRAIN_TF = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=5, translate=(0.03, 0.03)),
    T.ToTensor(),
    T.Normalize(IM_MEAN, IM_STD),
])

EVAL_TF = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IM_MEAN, IM_STD),
])

class OCTDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.tf = TRAIN_TF if train else EVAL_TF

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        with Image.open(r.filepath) as im:
            im = im.convert("L")
            x = self.tf(im)
        return x, int(r.Label), int(r.row_id), str(r.Patient_ID)

def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(df, train=False, seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        OCTDataset(df, train=train),
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
        generator=g if train else None,
        persistent_workers=False,
    )

print("Data pipeline ready.")

Data pipeline ready.


In [8]:
# =========================
# 8. Models, deterministic training, inference
# =========================
def make_resnet50():
    w = tvm.ResNet50_Weights.DEFAULT if PRETRAINED else None
    m = tvm.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m

def make_convnext_tiny():
    w = tvm.ConvNeXt_Tiny_Weights.DEFAULT if PRETRAINED else None
    m = tvm.convnext_tiny(weights=w)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m

MODEL_FACTORIES = {"ResNet50": make_resnet50, "ConvNeXtTiny": make_convnext_tiny}

# Preflight pretrained weights before training.
for name in RUN_ARCHITECTURES:
    try:
        m = MODEL_FACTORIES[name]()
        del m
    except Exception as e:
        raise RuntimeError(
            f"Could not initialize pretrained weights for {name}. Enable Internet on the first "
            f"Kaggle run or provide cached torchvision weights. Original error: {repr(e)}"
        )
print("Pretrained-weight preflight: PASSED")

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

@torch.no_grad()
def predict(model, loader):
    rows = []
    model.eval()
    for x, y, row_id, patient in loader:
        x = x.to(DEVICE, non_blocking=True)
        with autocast_ctx():
            logits = model(x)
        p = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()
        pred = (p >= 0.5).astype(int)
        for yy, pp, pr, rid, pat in zip(y.numpy(), p, pred, row_id.numpy(), patient):
            rows.append({
                "row_id": int(rid),
                "Patient_ID": str(pat),
                "Label": int(yy),
                "prob": float(pp),
                "pred": int(pr),
            })
    return pd.DataFrame(rows)

def train_one(model_name, train_df, val_df, fold, arm):
    # Same seed for paired arms within a fold and architecture.
    model_offset = 0 if model_name == "ResNet50" else 10
    run_seed = SEED + fold * 100 + model_offset
    seed_everything(run_seed)

    model = MODEL_FACTORIES[model_name]().to(DEVICE)
    trL = make_loader(train_df, train=True, seed=run_seed)
    vaL = make_loader(val_df, train=False, seed=run_seed)

    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    loss_fn = nn.CrossEntropyLoss()

    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        except TypeError:
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_auc = -np.inf
    best_state = None
    best_epoch = None
    wait = 0
    hist = []

    print(f"{model_name} fold={fold} arm={arm} train={len(train_df)} val={len(val_df)}")
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        n = 0

        for x, y, _, _ in trL:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with autocast_ctx():
                logits = model(x)
                loss = loss_fn(logits, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            total_loss += float(loss.detach()) * len(y)
            n += len(y)

        sch.step()
        vp = predict(model, vaL)
        vauc = roc_auc_score(vp.Label, vp.prob) if vp.Label.nunique() == 2 else np.nan

        hist.append({
            "epoch": ep,
            "train_loss": total_loss / max(n, 1),
            "val_auc": vauc,
        })
        print(
            f"  epoch {ep:02d} loss={total_loss/max(n,1):.4f} "
            f"valAUC={vauc:.4f} time={time.time()-t0:.0f}s"
        )

        if np.isfinite(vauc) and vauc > best_auc + 1e-6:
            best_auc = vauc
            best_epoch = ep
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                print("  early stop")
                break

    if best_state is None:
        raise RuntimeError(f"No valid validation checkpoint: {model_name} fold {fold} {arm}")

    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(hist), int(best_epoch)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 202MB/s]


Pretrained-weight preflight: PASSED


In [9]:
# =========================
# 9. Dependency-free Grad-CAM + Border Activation Ratio
# =========================
def resnet_gradcam_batch(model, x, targets):
    """
    Grad-CAM from ResNet-50 layer4 output.
    targets: integer class target per image. We use the TRUE binary class so both paired
    models are explained for the same target.
    Returns normalized CAMs [B,H,W] in [0,1].
    """
    activations = {}
    gradients = {}

    layer = model.layer4

    def fwd_hook(module, inp, out):
        activations["value"] = out

    def bwd_hook(module, grad_input, grad_output):
        gradients["value"] = grad_output[0]

    h1 = layer.register_forward_hook(fwd_hook)
    h2 = layer.register_full_backward_hook(bwd_hook)

    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits.gather(1, targets.view(-1, 1)).sum()
    score.backward()

    A = activations["value"]      # B,C,h,w
    G = gradients["value"]        # B,C,h,w
    w = G.mean(dim=(2,3), keepdim=True)
    cam = torch.relu((w * A).sum(dim=1, keepdim=True))
    cam = torch.nn.functional.interpolate(
        cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False
    ).squeeze(1)

    flat = cam.flatten(1)
    mn = flat.min(dim=1).values[:, None, None]
    mx = flat.max(dim=1).values[:, None, None]
    cam = (cam - mn) / (mx - mn + 1e-8)

    h1.remove()
    h2.remove()
    return cam.detach()

def border_ratio(cam, frac=XAI_BORDER_FRAC):
    # Fraction of total Grad-CAM mass located in outer frac border.
    B, H, W = cam.shape
    bh = max(1, int(round(H * frac)))
    bw = max(1, int(round(W * frac)))
    mask = torch.zeros((H, W), device=cam.device, dtype=cam.dtype)
    mask[:bh, :] = 1
    mask[-bh:, :] = 1
    mask[:, :bw] = 1
    mask[:, -bw:] = 1
    total = cam.sum(dim=(1,2)) + 1e-8
    border = (cam * mask).sum(dim=(1,2))
    return (border / total).detach().cpu().numpy()

def denormalize_tensor(x):
    mean = torch.tensor((0.485,0.456,0.406), device=x.device).view(3,1,1)
    std = torch.tensor((0.229,0.224,0.225), device=x.device).view(3,1,1)
    return torch.clamp(x * std + mean, 0, 1)

def save_overlay(x, cam, path, title):
    img = denormalize_tensor(x.detach()).cpu().permute(1,2,0).numpy()
    # Grayscale visual base.
    base = img.mean(axis=2)
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(base, cmap="gray")
    ax.imshow(cam.detach().cpu().numpy(), cmap="jet", alpha=0.42, vmin=0, vmax=1)
    ax.axis("off")
    ax.set_title(title, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def run_gradcam_oof(model, test_df, fold, arm):
    loader = make_loader(test_df, train=False, seed=SEED + fold)
    rows = []
    model.eval()

    for x, y, row_id, patient in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # Do not use autocast for XAI gradients; full precision improves numerical stability.
        cams = resnet_gradcam_batch(model, x, y)
        br = border_ratio(cams)

        for j, (rid, pat, yy, ratio) in enumerate(zip(row_id.numpy(), patient, y.cpu().numpy(), br)):
            rows.append({
                "row_id": int(rid),
                "Patient_ID": str(pat),
                "Label": int(yy),
                "fold": int(fold),
                "arm": arm,
                "border_ratio": float(ratio),
            })

            if int(rid) in visual_case_ids:
                disease = str(test_df.loc[test_df.row_id.astype(int) == int(rid), "Disease"].iloc[0])
                save_overlay(
                    x[j], cams[j],
                    XAI_DIR / f"row{int(rid)}_{disease}_fold{fold}_{arm}.png",
                    f"{disease} | fold {fold} | {arm}"
                )

    return pd.DataFrame(rows)

In [10]:
# =========================
# 10. Final deterministic paired rerun + in-run corrected XAI
# =========================
all_oof = []
all_xai = []
training_records = []

for model_name in RUN_ARCHITECTURES:
    print("\n" + "="*90)
    print("FINAL PATCH ARCHITECTURE:", model_name)
    print("="*90)

    for fold in range(1, FOLDS + 1):
        D = FOLD_DATA[fold]
        test = D["test"].copy()
        arm_preds = {}

        for arm, key in [("rigorous", "rig"), ("controlled_leak", "leak")]:
            # Final-patch notebook intentionally retrains BOTH arms.
            # This is required for the deterministic-reproducibility claim.
            model, hist, best_epoch = train_one(
                model_name, D[key], D["val"], fold, arm
            )

            pp = predict(model, make_loader(test, train=False, seed=SEED + fold))
            pp.to_csv(PRED_DIR / f"{model_name}_fold{fold}_{arm}.csv", index=False)
            hist.to_csv(PRED_DIR / f"{model_name}_fold{fold}_{arm}_history.csv", index=False)

            arm_preds[arm] = pp
            training_records.append({
                "architecture": model_name,
                "fold": fold,
                "arm": arm,
                "best_epoch": best_epoch,
                "train_n": len(D[key]),
                "val_n": len(D["val"]),
                "test_n": len(test),
            })

            # Quantitative XAI is limited to ResNet-50 to keep computation small.
            if model_name == XAI_ARCHITECTURE:
                xdf = run_gradcam_oof(model, test, fold, arm)
                all_xai.append(xdf)

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        r = arm_preds["rigorous"].rename(columns={"prob":"prob_rig","pred":"pred_rig"})
        l = arm_preds["controlled_leak"][["row_id","prob","pred"]].rename(
            columns={"prob":"prob_leak","pred":"pred_leak"}
        )

        merged = test[[
            "row_id","Patient_ID","Label","Disease","sample_key",
            "n_train_siblings","n_same_disease_siblings"
        ]].merge(
            r[["row_id","prob_rig","pred_rig"]], on="row_id", how="inner"
        ).merge(
            l, on="row_id", how="inner"
        )

        assert len(merged) == len(test)
        merged["fold"] = fold
        merged["architecture"] = model_name
        all_oof.append(merged)

oof = pd.concat(all_oof, ignore_index=True)
oof.to_csv(OUT / "oof_paired_predictions.csv", index=False)

training_log = pd.DataFrame(training_records)
training_log.to_csv(OUT / "training_summary.csv", index=False)

xai_long = pd.concat(all_xai, ignore_index=True)
xai_long.to_csv(XAI_DIR / "gradcam_border_ratio_long.csv", index=False)

print("\nOOF rows:", len(oof))
print("XAI rows:", len(xai_long))


FINAL PATCH ARCHITECTURE: ResNet50
ResNet50 fold=1 arm=rigorous train=652 val=167
  epoch 01 loss=0.6006 valAUC=0.9622 time=9s
  epoch 02 loss=0.2494 valAUC=0.9856 time=7s
  epoch 03 loss=0.1262 valAUC=0.9875 time=7s
  epoch 04 loss=0.0760 valAUC=0.9957 time=7s
  epoch 05 loss=0.0595 valAUC=0.9830 time=7s
  epoch 06 loss=0.0566 valAUC=0.9924 time=7s
  epoch 07 loss=0.0554 valAUC=0.9967 time=7s
  epoch 08 loss=0.0386 valAUC=0.9967 time=7s
  epoch 09 loss=0.0223 valAUC=0.9967 time=7s
  epoch 10 loss=0.0200 valAUC=0.9976 time=7s
  epoch 11 loss=0.0154 valAUC=0.9976 time=7s
  epoch 12 loss=0.0199 valAUC=0.9976 time=7s
ResNet50 fold=1 arm=controlled_leak train=652 val=167
  epoch 01 loss=0.6034 valAUC=0.9288 time=7s
  epoch 02 loss=0.2486 valAUC=0.9919 time=7s
  epoch 03 loss=0.1458 valAUC=0.9968 time=7s
  epoch 04 loss=0.0865 valAUC=0.9951 time=7s
  epoch 05 loss=0.0862 valAUC=0.9986 time=7s
  epoch 06 loss=0.0461 valAUC=0.9996 time=7s
  epoch 07 loss=0.0458 valAUC=0.9988 time=7s
  epoch 

In [11]:
# =========================
# 11. Patient-cluster paired performance statistics
# =========================
def metric_from_arrays(y, p, metric):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pred = (p >= 0.5).astype(int)

    if metric == "AUC":
        return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan
    if metric == "PR_AUC":
        return average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan
    if metric == "ACC":
        return accuracy_score(y, pred)
    if metric == "Macro_F1":
        return f1_score(y, pred, average="macro", zero_division=0)
    if metric == "Balanced_ACC":
        return balanced_accuracy_score(y, pred)
    if metric == "MCC":
        return matthews_corrcoef(y, pred)
    if metric == "Sensitivity":
        return recall_score(y, pred, pos_label=1, zero_division=0)
    if metric == "Specificity":
        return recall_score(y, pred, pos_label=0, zero_division=0)
    if metric == "Brier":
        return brier_score_loss(y, p)
    if metric == "LogLoss":
        return log_loss(y, np.c_[1-p, p], labels=[0,1])
    raise KeyError(metric)

METRICS = [
    "AUC","PR_AUC","ACC","Macro_F1","Balanced_ACC",
    "MCC","Sensitivity","Specificity","Brier","LogLoss"
]

def macro_fold_metric(d, prob_col, metric):
    vals = []
    for _, g in d.groupby("fold"):
        v = metric_from_arrays(g.Label.to_numpy(), g[prob_col].to_numpy(), metric)
        if np.isfinite(v):
            vals.append(v)
    return float(np.mean(vals)) if vals else np.nan

def paired_cluster_bootstrap(d, metric, B=BOOTSTRAPS, seed=SEED):
    patients = np.array(sorted(d.Patient_ID.astype(str).unique()))
    byp = {p: d[d.Patient_ID.astype(str) == p] for p in patients}
    rng = np.random.default_rng(seed)
    deltas = []

    for _ in range(B):
        samp = rng.choice(patients, size=len(patients), replace=True)
        x = pd.concat([byp[p] for p in samp], ignore_index=True)
        rv = macro_fold_metric(x, "prob_rig", metric)
        lv = macro_fold_metric(x, "prob_leak", metric)
        if np.isfinite(rv) and np.isfinite(lv):
            deltas.append(lv-rv)

    if len(deltas) < max(200, B//2):
        raise RuntimeError(f"Too few bootstrap replicates for {metric}")

    return float(np.percentile(deltas,2.5)), float(np.percentile(deltas,97.5)), len(deltas)

def cluster_randomization_p(d, metric, B=PERMUTATIONS, seed=SEED+7):
    obs = macro_fold_metric(d,"prob_leak",metric)-macro_fold_metric(d,"prob_rig",metric)
    pats = np.array(sorted(d.Patient_ID.astype(str).unique()))
    rng = np.random.default_rng(seed)
    extreme = 0
    valid = 0

    for _ in range(B):
        swap = set(pats[rng.integers(0,2,size=len(pats)).astype(bool)])
        x = d.copy()
        mask = x.Patient_ID.astype(str).isin(swap).to_numpy()

        a = x.prob_rig.to_numpy().copy()
        b = x.prob_leak.to_numpy().copy()
        aa = a.copy(); bb = b.copy()
        aa[mask] = b[mask]
        bb[mask] = a[mask]
        x["prob_rig"] = aa
        x["prob_leak"] = bb

        dd = macro_fold_metric(x,"prob_leak",metric)-macro_fold_metric(x,"prob_rig",metric)
        if np.isfinite(dd):
            valid += 1
            if abs(dd) >= abs(obs)-1e-15:
                extreme += 1

    return float((extreme+1)/(valid+1))

rows = []
for arch, d in oof.groupby("architecture"):
    for metric in METRICS:
        rv = macro_fold_metric(d,"prob_rig",metric)
        lv = macro_fold_metric(d,"prob_leak",metric)
        lo, hi, bn = paired_cluster_bootstrap(
            d, metric, seed=SEED+sum(map(ord,arch+metric))
        )
        p = cluster_randomization_p(
            d, metric, seed=SEED+123+sum(map(ord,arch+metric))
        )

        rows.append({
            "Architecture": arch,
            "Metric": metric,
            "Rigorous": rv,
            "ControlledLeak": lv,
            "Delta_LeakMinusRig": lv-rv,
            "Delta_CI_Lo": lo,
            "Delta_CI_Hi": hi,
            "ClusterRandomization_p": p,
            "BootstrapReplicates": bn,
        })

stats = pd.DataFrame(rows)
stats.to_csv(OUT / "cluster_aware_metrics.csv", index=False)
print(stats.to_string(index=False, float_format=lambda x:f"{x:.4f}"))

Architecture       Metric  Rigorous  ControlledLeak  Delta_LeakMinusRig  Delta_CI_Lo  Delta_CI_Hi  ClusterRandomization_p  BootstrapReplicates
ConvNeXtTiny          AUC    0.9935          0.9938              0.0003      -0.0014       0.0022                  0.8741                 2000
ConvNeXtTiny       PR_AUC    0.9947          0.9950              0.0003      -0.0010       0.0015                  0.8651                 2000
ConvNeXtTiny          ACC    0.9792          0.9808              0.0017      -0.0050       0.0088                  0.8241                 2000
ConvNeXtTiny     Macro_F1    0.9792          0.9808              0.0017      -0.0051       0.0090                  0.7391                 2000
ConvNeXtTiny Balanced_ACC    0.9792          0.9808              0.0017      -0.0053       0.0092                  0.8081                 2000
ConvNeXtTiny          MCC    0.9587          0.9622              0.0035      -0.0109       0.0173                  0.5127                 2000

In [12]:
# =========================
# 12. Corrected Grad-CAM quantitative analysis
# =========================
# Pair rigorous/leak BR by row_id.
xr = xai_long[xai_long.arm == "rigorous"].rename(
    columns={"border_ratio":"border_ratio_rig"}
)
xl = xai_long[xai_long.arm == "controlled_leak"][["row_id","border_ratio"]].rename(
    columns={"border_ratio":"border_ratio_leak"}
)

xpair = xr[["row_id","Patient_ID","Label","fold","border_ratio_rig"]].merge(
    xl, on="row_id", how="inner"
).merge(
    cohort[["row_id","Disease","sample_key"]], on="row_id", how="left"
)

assert len(xpair) == len(cohort), (
    f"Expected one paired XAI record for every cohort image ({len(cohort)}), got {len(xpair)}"
)
xpair["delta_border_ratio"] = xpair.border_ratio_leak - xpair.border_ratio_rig
xpair.to_csv(XAI_DIR / "gradcam_border_ratio_paired.csv", index=False)

def cluster_bootstrap_mean_delta(df, value_col, B=BOOTSTRAPS, seed=SEED+900):
    pats = np.array(sorted(df.Patient_ID.astype(str).unique()))
    byp = {p: df[df.Patient_ID.astype(str) == p] for p in pats}
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(B):
        samp = rng.choice(pats, len(pats), replace=True)
        x = pd.concat([byp[p] for p in samp], ignore_index=True)
        vals.append(float(x[value_col].mean()))
    return (
        float(df[value_col].mean()),
        float(np.percentile(vals,2.5)),
        float(np.percentile(vals,97.5))
    )

xrows = []
groups = [("ALL", xpair)] + [(d, g) for d,g in xpair.groupby("Disease")]
for name, g in groups:
    mean_delta, lo, hi = cluster_bootstrap_mean_delta(
        g, "delta_border_ratio",
        seed=SEED+900+sum(map(ord,str(name)))
    )
    xrows.append({
        "Disease": name,
        "n_images": len(g),
        "n_patients": g.Patient_ID.nunique(),
        "Rigorous_BR_mean": float(g.border_ratio_rig.mean()),
        "ControlledLeak_BR_mean": float(g.border_ratio_leak.mean()),
        "Delta_BR_mean": mean_delta,
        "Delta_BR_CI_Lo": lo,
        "Delta_BR_CI_Hi": hi,
    })

xstats = pd.DataFrame(xrows)
xstats.to_csv(XAI_DIR / "gradcam_border_ratio_cluster_summary.csv", index=False)

print("\n=== CORRECTED GRAD-CAM BORDER-RATIO SUMMARY ===")
print(xstats.to_string(index=False, float_format=lambda x:f"{x:.4f}"))

print(
    "\nInterpretation: Grad-CAM BR is a secondary attribution analysis. "
    "Do not describe it as proof of a causal memorization mechanism."
)


=== CORRECTED GRAD-CAM BORDER-RATIO SUMMARY ===
Disease  n_images  n_patients  Rigorous_BR_mean  ControlledLeak_BR_mean  Delta_BR_mean  Delta_BR_CI_Lo  Delta_BR_CI_Hi
    ALL      1200         855            0.0385                  0.0362        -0.0023         -0.0054          0.0005
    CNV       406         212            0.0264                  0.0312         0.0048          0.0021          0.0072
    DME       109          82            0.0661                  0.0526        -0.0135         -0.0343          0.0053
 DRUSEN        85          72            0.1109                  0.1023        -0.0086         -0.0321          0.0108
 NORMAL       600         528            0.0315                  0.0272        -0.0043         -0.0068         -0.0018

Interpretation: Grad-CAM BR is a secondary attribution analysis. Do not describe it as proof of a causal memorization mechanism.


In [13]:
# =========================
# 13. Final integrity report
# =========================
summary = {
    "prior_output_reused_for": [
        "frozen cohort manifest",
        "fixed test assignments",
        "rigorous training manifests",
        "safe validation manifests",
    ],
    "old_predictions_reused": False,
    "old_model_checkpoints_reused": False,
    "old_xai_reused": False,
    "cohort_images": int(len(cohort)),
    "cohort_patients": int(cohort.Patient_ID.nunique()),
    "decoded_content_duplicates": int(cohort.duplicated("decoded_fp").sum()),
    "folds": FOLDS,
    "architectures": RUN_ARCHITECTURES,
    "pathology_subtype_counts_exactly_matched_every_fold": bool(
        all(
            (audit_df[f"rig_{d}"] == audit_df[f"leak_{d}"]).all()
            for d in DISEASE_ORDER
        )
    ),
    "rigorous_train_test_patient_overlap_zero": bool(
        audit_df.rig_train_test_patient_overlap.eq(0).all()
    ),
    "exact_or_decoded_test_content_overlap_zero": bool(
        (
            audit_df.rig_train_test_decoded_overlap.eq(0)
            & audit_df.leak_train_test_decoded_overlap.eq(0)
        ).all()
    ),
    "deterministic_settings": versions,
    "xai_architecture": XAI_ARCHITECTURE,
    "xai_target": "true binary class",
    "xai_border_fraction": XAI_BORDER_FRAC,
    "xai_quantitative_cases": int(len(xpair)),
    "xai_visual_case_selection": "seeded disease-stratified, selected before predictions",
}

with open(OUT / "final_integrity_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

assert summary["decoded_content_duplicates"] == 0
assert summary["pathology_subtype_counts_exactly_matched_every_fold"]
assert summary["rigorous_train_test_patient_overlap_zero"]
assert summary["exact_or_decoded_test_content_overlap_zero"]
assert summary["old_predictions_reused"] is False
assert summary["old_model_checkpoints_reused"] is False

primary = stats[stats.Metric.isin(["AUC","PR_AUC","Macro_F1","ACC","Brier"])].copy()
primary.to_csv(OUT / "manuscript_primary_result_table.csv", index=False)

print("\n=== MANUSCRIPT-READY PRIMARY RESULTS ===")
print(
    primary[[
        "Architecture","Metric","Rigorous","ControlledLeak",
        "Delta_LeakMinusRig","Delta_CI_Lo","Delta_CI_Hi",
        "ClusterRandomization_p"
    ]].to_string(index=False, float_format=lambda x:f"{x:.4f}")
)

{
  "prior_output_reused_for": [
    "frozen cohort manifest",
    "fixed test assignments",
    "rigorous training manifests",
    "safe validation manifests"
  ],
  "old_predictions_reused": false,
  "old_model_checkpoints_reused": false,
  "old_xai_reused": false,
  "cohort_images": 1200,
  "cohort_patients": 855,
  "decoded_content_duplicates": 0,
  "folds": 5,
  "architectures": [
    "ResNet50",
    "ConvNeXtTiny"
  ],
  "pathology_subtype_counts_exactly_matched_every_fold": true,
  "rigorous_train_test_patient_overlap_zero": true,
  "exact_or_decoded_test_content_overlap_zero": true,
  "deterministic_settings": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "cuda": "12.8",
    "device": "Tesla T4",
    "cudnn_deterministic": true,
    "cudnn_benchmark": false,
    "deterministic_algorithms": true
  },
  "xai_architecture": "ResNet50",
  "xai_target": "true binary class",
  "xai_border_fraction": 0.1,
  "xai_quantitative_cases": 120

In [14]:
# =========================
# 14. Bundle final results
# =========================
readme = f"""OCT FINAL PATCH — SUBTYPE MATCHING + DETERMINISTIC RERUN + CORRECTED XAI

Inputs:
- Prior corrected OCT notebook output: {PRIOR}
- Raw OCT dataset root: {ROOT}

Final design:
- Frozen 1,200-image cohort and fixed test/validation assignments reused from prior corrected run.
- NO prior model predictions/checkpoints/XAI reused.
- Rigorous and controlled-leak arms rerun under deterministic settings.
- Controlled-leak train uses exact Disease-matched swaps:
  NORMAL<->NORMAL, CNV<->CNV, DME<->DME, DRUSEN<->DRUSEN.
- Exact/decoded-content test images forbidden from both arms.
- Rigorous arm has zero train/test patient overlap.
- ResNet-50 and ConvNeXt-Tiny rerun.
- Grad-CAM quantitative BR analysis on all ResNet-50 OOF test images.
- Visual XAI cases selected before predictions with seeded disease-stratified sampling.
- Patient-cluster bootstrap CIs used for performance and XAI effect estimates.

Key outputs:
- subtype_matched_leakage_audit_by_fold.csv
- oof_paired_predictions.csv
- cluster_aware_metrics.csv
- training_summary.csv
- manuscript_primary_result_table.csv
- final_integrity_summary.json
- splits/
- predictions/
- xai/gradcam_border_ratio_long.csv
- xai/gradcam_border_ratio_paired.csv
- xai/gradcam_border_ratio_cluster_summary.csv
- xai/preselected_visual_cases.csv
- xai/*overlay PNGs
"""

(OUT / "README.txt").write_text(readme)

archive = BASE / "OCT_FinalPatch_Results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(BASE))

print("Final archive:", archive)
print("FINAL PATCH COMPLETE.")

Final archive: /kaggle/working/OCT_FinalPatch_Results.zip
FINAL PATCH COMPLETE.
